# RFE 전 1차 변수 골라내기

In [1]:
import duckdb
import os

# 1. 파일 경로 설정
base_path = r'C:\Workspace\06_ML_projdect\26_1_COIN\data\rfe_sample_data'
train_src = os.path.join(base_path, 'rfe_sample_train.parquet')
test_src = os.path.join(base_path, 'rfe_sample_test.parquet')

train_dst = os.path.join(base_path, 'rfe_train.parquet')
test_dst = os.path.join(base_path, 'rfe_test.parquet')

# 2. 제거할 컬럼 리스트 (나열해주신 변수들)
exclude_cols = [
    "is_warmup_28d", "total_seeks_28d_ewma", "total_seeks_28d_dai", "total_seeks_28d_cid", "total_seeks_28d_zscore",
    "total_seeks_28d_std", "total_seeks_28d_mean", "total_seeks_28d_sum", "total_seeks_14d_ewma", "total_seeks_14d_dai",
    "total_seeks_14d_cid", "total_seeks_14d_asfd", "total_seeks_14d_zscore", "total_seeks_14d_std", "total_seeks_14d_sum",
    "total_seeks_14d_max", "total_seeks_7d_ewma", "total_seeks_7d_dai", "total_seeks_7d_cid", "total_seeks_7d_asfd",
    "total_seeks_7d_zscore", "total_seeks_7d_mean", "total_seeks_7d_sum", "total_seeks_7d_max", "total_reads_28d_ewma",
    "total_reads_28d_dai", "total_reads_28d_cid", "total_reads_28d_asfd", "total_reads_28d_zscore", "total_reads_28d_std",
    "total_reads_28d_mean", "total_reads_28d_sum", "total_reads_28d_max", "total_reads_14d_ewma", "total_reads_14d_dai",
    "total_reads_14d_cid", "total_reads_14d_asfd", "total_reads_14d_zscore", "total_reads_14d_std", "total_reads_14d_mean",
    "total_reads_14d_sum", "total_reads_14d_max", "total_reads_7d_ewma", "total_reads_7d_dai", "total_reads_7d_cid",
    "total_reads_7d_asfd", "total_reads_7d_zscore", "total_reads_7d_std", "total_reads_7d_mean", "total_reads_7d_sum",
    "total_reads_7d_max", "s242_28d_ewma", "s242_28d_dai", "s242_28d_cid", "s242_28d_zscore", "s242_28d_mean",
    "s242_14d_ewma", "s242_14d_dai", "s242_14d_cid", "s242_14d_asfd", "s242_14d_zscore", "s242_14d_std",
    "s242_14d_mean", "s242_14d_sum", "s242_14d_max", "s242_7d_ewma", "s242_7d_dai", "s242_7d_cid", "s242_7d_asfd",
    "s242_7d_zscore", "s242_7d_std", "s242_7d_mean", "s242_7d_sum", "s242_7d_max", "s241_28d_ewma", "s241_28d_dai",
    "s241_28d_cid", "s241_28d_zscore", "s241_28d_sum", "s241_14d_ewma", "s241_14d_dai", "s241_14d_cid",
    "s241_14d_asfd", "s241_14d_zscore", "s241_14d_std", "s241_14d_mean", "s241_14d_sum", "s241_14d_max",
    "s241_7d_ewma", "s241_7d_dai", "s241_7d_cid", "s241_7d_asfd", "s241_7d_zscore", "s241_7d_std", "s241_7d_mean",
    "s241_7d_sum", "s241_7d_max", "timeout_severity_ratio", "s190_28d_ewma", "s190_28d_dai", "s190_28d_cid",
    "s190_28d_asfd", "s190_28d_zscore", "s190_28d_std", "s190_28d_max", "s190_14d_ewma", "s190_14d_dai",
    "s190_14d_cid", "s190_14d_asfd", "s190_14d_zscore", "s190_14d_std", "s190_14d_mean", "s190_14d_max",
    "s190_7d_ewma", "s190_7d_dai", "s190_7d_cid", "s190_7d_asfd", "s190_7d_zscore", "s190_7d_std", "s190_7d_mean",
    "s194_over40_7d_count", "s194_28d_ewma", "s194_28d_dai", "s194_28d_cid", "s194_28d_asfd", "s194_28d_zscore",
    "s194_28d_std", "s194_28d_mean", "s194_28d_max", "s194_14d_ewma", "s194_14d_dai", "s194_14d_cid", "s194_14d_asfd",
    "s194_14d_zscore", "s194_14d_std", "s194_14d_mean", "s194_14d_max", "s194_7d_ewma", "s194_7d_dai", "s194_7d_cid",
    "s194_7d_asfd", "s194_7d_zscore", "s194_7d_std", "s194_7d_max", "s197_damaged"
]

# EXCLUDE 구문에 사용할 문자열 생성
exclude_str = ", ".join(exclude_cols)

con = duckdb.connect()

# 3. 데이터 처리 및 저장
for src, dst in [(train_src, train_dst), (test_src, test_dst)]:
    print(f"처리 중: {os.path.basename(src)} -> {os.path.basename(dst)}")
    
    # 실제 파일에 존재하는 컬럼만 제외하기 위해 DuckDB에서 컬럼 목록 확인 후 처리
    cols_in_file = con.execute(f"SELECT * FROM read_parquet('{src}') LIMIT 0").df().columns.tolist()
    final_exclude = [c for c in exclude_cols if c in cols_in_file]
    final_exclude_str = ", ".join(final_exclude)

    if final_exclude:
        con.execute(f"""
            COPY (SELECT * EXCLUDE ({final_exclude_str}) FROM read_parquet('{src}'))
            TO '{dst}' (FORMAT PARQUET)
        """)
    else:
        con.execute(f"COPY (SELECT * FROM read_parquet('{src}')) TO '{dst}' (FORMAT PARQUET)")

con.close()
print("작업 완료: 변수가 제거된 파일이 저장되었습니다.")


처리 중: rfe_sample_train.parquet -> rfe_train.parquet
처리 중: rfe_sample_test.parquet -> rfe_test.parquet
작업 완료: 변수가 제거된 파일이 저장되었습니다.


In [3]:
# EDA

import duckdb
import pandas as pd
import time
"""
C:\Workspace\06_ML_projdect\26_1_COIN\data\rfe_sample_data\rfe_train.parquet
C:\Workspace\06_ML_projdect\26_1_COIN\data\rfe_sample_data\rfe_test.parquet
"""
# 파일 경로 지정
parquet_file = r"C:\Workspace\06_ML_projdect\26_1_COIN\data\rfe_sample_data\rfe_test.parquet"

# 판다스 출력 제한 해제 (모든 컬럼과 행을 숨김없이 표시)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 2000)

print(f"[{parquet_file}]")
print("종합 EDA 및 데이터 무결성 검증을 시작합니다 ...\n")
start_time = time.time()

# DuckDB 인메모리 연결
con = duckdb.connect()

try:
    # ---------------------------------------------------------
    # 1. 데이터 규격 (행/열 개수)
    # ---------------------------------------------------------
    print("=== [1. 데이터 규격 확인] ===")
    total_rows = con.execute(f"SELECT COUNT(*) FROM read_parquet('{parquet_file}')").fetchone()[0]
    schema_df = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{parquet_file}')").fetchdf()
    col_names = schema_df['column_name'].tolist()
    
    print(f"총 행 수(Rows): {total_rows:,} 개")
    print(f"총 열 수(Columns): {len(col_names)} 개\n")

    # ---------------------------------------------------------
    # 2. 상위 10개 데이터 샘플 (모든 열 표시)
    # ---------------------------------------------------------
    print("=== [2. 상위 10개 데이터 샘플 (생략 없음)] ===")
    sample_df = con.execute(f"SELECT * FROM read_parquet('{parquet_file}') LIMIT 10").fetchdf()
    print(sample_df)
    print("\n")

    # ---------------------------------------------------------
    # 3. 하드디스크 개체 및 클래스 분포 통계 (ML Target 반영 수정 완료)
    # ---------------------------------------------------------
    print("=== [3. 하드디스크 개체 및 클래스 분포 통계] ===")
    status_query = f"""
        SELECT 
            COUNT(DISTINCT serial_number) AS total_objects,
            COUNT(DISTINCT CASE WHEN failure = 1 THEN serial_number ELSE NULL END) AS failed_objects,
            COUNT(*) AS total_rows,
            SUM(CAST(failure AS INTEGER)) AS target_1_rows,
            SUM(CASE WHEN failure = 0 THEN 1 ELSE 0 END) AS target_0_rows
        FROM read_parquet('{parquet_file}')
    """
    stats = con.execute(status_query).fetchdf().iloc[0]

    total_obj = stats['total_objects']
    failed_obj = stats['failed_objects']
    healthy_obj = total_obj - failed_obj
    
    print("[개체 단위 통계 (물리적인 하드디스크 개수)]")
    print(f"- 전체 고유 개체 수: {int(total_obj):,} 개")
    print(f"- 정상 작동 하드: {int(healthy_obj):,} 개 ({(healthy_obj/total_obj)*100:.2f}%)")
    print(f"- 고장 발생 개체: {int(failed_obj):,} 개 ({(failed_obj/total_obj)*100:.2f}%)")
    print(f"- 개체 단위 비율 (Class 1 : 0) = 1 : {healthy_obj / failed_obj:.2f}\n")

    print("[행 단위 클래스 분포 (❗진짜 ML 모델이 학습할 Target 레이블 비율)]")
    total_r = stats['total_rows']
    target_1 = stats['target_1_rows']
    target_0 = stats['target_0_rows']
    
    print(f"- 총 데이터 행 수: {int(total_r):,} 개")
    print(f"- Class 0 (정상인 날): {int(target_0):,} 개 ({target_0/total_r*100:.2f}%)")
    print(f"- Class 1 (고장 임박): {int(target_1):,} 개 ({target_1/total_r*100:.2f}%)")
    
    if target_1 > 0:
        ratio = target_0 / target_1
        print(f"- 실제 타겟 데이터 불균형 비율 (Class 1 : 0) = 1 : {ratio:.1f}\n")

    # # ---------------------------------------------------------
    # # 4. 모든 열에 대한 결측치(NULL) 개수 세기 (전체 열 표시)
    # # ---------------------------------------------------------
    # print("=== [4. 컬럼별 결측치 집계 (전체 열)] ===")
    # null_count_selects = [f"COUNT(*) - COUNT(\"{col}\") AS \"{col}\"" for col in col_names]
    # query_nulls = f"SELECT {', '.join(null_count_selects)} FROM read_parquet('{parquet_file}')"
    # null_counts = con.execute(query_nulls).fetchdf().iloc[0]
    
    # null_summary = pd.DataFrame({'Missing_Count': null_counts})
    # null_summary['Missing_Ratio(%)'] = (null_summary['Missing_Count'] / total_rows) * 100
    
    # # 필터링 없이 정렬만 수행하여 모든 열을 보여줌
    # null_summary_sorted = null_summary.sort_values(by='Missing_Count', ascending=False)
    
    # pd.set_option('display.max_rows', None)
    # print(null_summary_sorted)
    # pd.reset_option('display.max_rows')
    # print("\n")

    # ---------------------------------------------------------
    # 5. 열 별 간단한 기초 통계 (최솟값, 최댓값, 평균, 표준편차)
    # ---------------------------------------------------------
    print("=== [5. 열 별 간단한 기초 통계 (Numeric Data)] ===")
    summary_df = con.execute(f"SUMMARIZE SELECT * FROM read_parquet('{parquet_file}')").fetchdf()
    stats_df = summary_df[['column_name', 'column_type', 'min', 'max', 'avg', 'std']].copy()
    
    pd.set_option('display.max_rows', None)
    print(stats_df)
    pd.reset_option('display.max_rows')
    print("\n")

    # # ---------------------------------------------------------
    # # 6. 시계열 연속성 검사 (Date Gap)
    # # ---------------------------------------------------------
    # print("=== [6. 시계열 연속성(Date Gap) 검사] ===")
    # gap_check_query = f"""
    #     WITH DateRange AS (
    #         SELECT 
    #             serial_number,
    #             MIN(CAST(date AS DATE)) as start_date,
    #             MAX(CAST(date AS DATE)) as end_date,
    #             COUNT(*) as actual_row_count,
    #             (MAX(CAST(date AS DATE)) - MIN(CAST(date AS DATE)) + 1) as expected_row_count
    #         FROM read_parquet('{parquet_file}')
    #         GROUP BY serial_number
    #     )
    #     SELECT 
    #         COUNT(*) AS serials_with_gaps,
    #         SUM(expected_row_count - actual_row_count) AS total_missing_days
    #     FROM DateRange
    #     WHERE actual_row_count != expected_row_count
    # """
    # gap_result = con.execute(gap_check_query).fetchdf().iloc[0]
    
    # if gap_result['serials_with_gaps'] == 0:
    #     print("✅ 모든 개체의 날짜가 하루도 빠짐없이 연속적입니다.")
    # else:
    #     print(f"⚠️ 날짜 공백(Gap)이 발견된 개체 수: {int(gap_result['serials_with_gaps']):,} 개")
    #     print(f"⚠️ 총 누락된 날짜(데이터 행) 수: {int(gap_result['total_missing_days']):,} 일")

except Exception as e:
    print(f"❌ 검증 중 오류 발생: {e}")
finally:
    con.close()
    pd.reset_option('display.max_columns')
    pd.reset_option('display.width')
    
    end_time = time.time()
    print(f"\n모든 종합 검증 완료. 총 소요 시간: {end_time - start_time:.2f}초")


<>:6: SyntaxWarning: invalid escape sequence '\W'
<>:6: SyntaxWarning: invalid escape sequence '\W'
C:\Users\joon6\AppData\Local\Temp\ipykernel_20072\3072688205.py:6: SyntaxWarning: invalid escape sequence '\W'
  """


[C:\Workspace\06_ML_projdect\26_1_COIN\data\rfe_sample_data\rfe_test.parquet]
종합 EDA 및 데이터 무결성 검증을 시작합니다 ...

=== [1. 데이터 규격 확인] ===
총 행 수(Rows): 74,998 개
총 열 수(Columns): 87 개

=== [2. 상위 10개 데이터 샘플 (생략 없음)] ===
  serial_number       date  failure  age_weighted_seek_error  age_weighted_workload  cumulative_error_score  error_growth_ratio  error_saturation_score  fatal_crash_interaction  firmware_struggle_index  io_asymmetry_index  late_stage_degradation  log_shock_fly_interaction  multi_error_coincidence  pending_to_offline_ratio  reallocated_pending_ratio  s187_error_rate  s198_error_rate  s199_error_density  seek_error_density  shock_fatigue_rate  shock_seek_interaction  temp_error_index  thermal_stress_index  timeout_read_density  timeout_seek_density  timeout_to_uncorrectable_lag1  workload_intensity  write_stability_ratio  is_warmup_7d  is_warmup_14d  s5_damaged  s187_damaged  s198_damaged  seek_damaged  timeout_5s_damaged  s5_ever_flag  s187_ever_flag  cascading_failure_flag  dat